# Flow matching for a handwritten digits dataset

In [ ]:
%load_ext autoreload
%autoreload 2

import embedding # Auxiliary module for dimensionality reduction
import models  # Auxiliary module for neural network architecture and training
import plotting  # Auxiliary module for visualizations

## Load data

We load a handwritten digits dataset from sklearn:

In [ ]:
from sklearn.datasets import load_digits
import numpy as np

data = load_digits()
target_data = data.images
target_data /= target_data.max()  # Normalize pixel values to [0, 1]
target_labels = data.target
# Shuffle data and labels together
perm = np.random.permutation(len(target_data))
target_data = target_data[perm]
target_labels = target_labels[perm]
n = len(target_data)
unique_labels = set(target_labels)

Here is a sample of the images we will try to replicate:

In [ ]:
plotting.plot_image_grid(target_data, target_labels)

## Train rectified flow

We start by generating sample from a gaussian distribution with the same shape as the images we want to generate. We also create independent couplings between the source and target distributions.

In [ ]:
import numpy as np

source_data = np.random.randn(*target_data.shape)

num_couplings = 100000
couplings = models.sample_independent_couplings(source_data, target_data, num_couplings=num_couplings, target_labels=target_labels)

We cannot directly visualize a space with so many dimensions, but we can plot a projection ot such space into 2D:

In [ ]:
source_data_2d, target_data_2d = embedding.embed_data(source_data, target_data)
plotting.plot_distributions(source_data_2d, target_data_2d, target_labels=target_labels)

Let's now traing a rectified flow:

In [ ]:
num_reflow_steps = 2
for i in range(num_reflow_steps):
    print(f"Reflow step {i + 1}/{num_reflow_steps}...")
    couplings, rectified_network = models.reflow(
        couplings, 
        model_arguments={"network": "unet", "num_epochs": 100, "verbose": True, "log_frequency": 1},
        simulation_arguments={"n_steps": 100}
    )

Show architecture of trained network

In [ ]:
plotting.plot_network(rectified_network, couplings, save_filename="network_architecture")

Once the flow is trained, we can generate some trajectories:

In [ ]:
source_data = np.random.randn(*target_data.shape)
source_labels = np.random.choice(list(unique_labels), size=source_data.shape[0])
trajectories = models.compute_trajectories(rectified_network, source_data, labels=source_labels, n_steps=200)

 The trajectories can be projected onto 2D space for visualization:

In [ ]:
target_data_2d, trajectories_2d = embedding.embed_data(target_data, trajectories[:200])
fig = plotting.animate_trajectories(trajectories_2d, labels=source_labels[:200], target_data=target_data_2d)
fig.update_layout(width=800, height=600, legend=dict(orientation="v", yanchor="top", xanchor="left", x=1, y=1))

And we can also visualize the generated images

In [ ]:
# Show synthetic images obtained
generated_endpoints = [traj[-1][1] for traj in trajectories]
plotting.plot_image_grid(generated_endpoints, source_labels)